In [ ]:
#This script is called utils.py

import os

import logging

import subprocess

import requests

import time

from multiprocessing import BoundedSemaphore



# Semaphore to limit concurrent requests

global_request_semaphore = BoundedSemaphore(3)  # Limit to 3 concurrent requests



def log_message(message, level="INFO"):

    levels = {

        "INFO": logging.INFO,

        "WARNING": logging.WARNING,

        "ERROR": logging.ERROR

    }

    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

    logging.log(levels.get(level, logging.INFO), message)



def create_directory(path):

    try:

        os.makedirs(path, exist_ok=True)

        return path

    except Exception as e:

        log_message(f"Error creating directory {path}: {e}", level="ERROR")

        return None



def fetch_numeric_id(accession, email="your_email@example.com", tool="FetchOmics", max_retries=10, delay=5):

    params = {

        "db": "gds",

        "term": f"{accession}[Accession]",

        "retmode": "json",

        "email": email,

        "tool": tool

    }



    for attempt in range(1, max_retries + 1):

        try:

            with global_request_semaphore:  # Limit concurrent requests

                response = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi", params=params)

            

            # Check for rate-limiting response (HTTP 429)

            if response.status_code == 429:

                retry_after = int(response.headers.get("Retry-After", delay))

                log_message(f"Rate limit exceeded (Attempt {attempt}). Retrying in {retry_after} seconds...")

                time.sleep(retry_after)

                continue



            response.raise_for_status()

            data = response.json()

            id_list = data.get("esearchresult", {}).get("idlist", [])

            if id_list:

                log_message(f"Accession: {accession}, First GDS ID: {id_list[0]}")

                time.sleep(0.34)  # Ensure at most 3 requests per second

                return id_list[0]

            else:

                log_message(f"Accession: {accession}, No GDS ID found.", level="WARNING")

                return None

        except requests.exceptions.RequestException as e:

            log_message(f"Attempt {attempt}: Error fetching numeric ID for {accession}: {e}", level="WARNING")

            if attempt < max_retries:

                backoff_time = delay * (2 ** (attempt - 1))

                log_message(f"Retrying in {backoff_time} seconds...")

                time.sleep(backoff_time)

            else:

                log_message(f"Max retries reached for {accession}.", level="ERROR")

                return None



def fetch_runinfo(gds_id, output_dir, max_retries=10, delay=5):

    os.makedirs(output_dir, exist_ok=True)

    output_file = os.path.join(output_dir, f"{gds_id}_SRA_RunInfo.csv")



    command = (

        f"elink -db gds -id {gds_id} -target sra | "

        f"efetch -format runinfo > {output_file}"

    )



    for attempt in range(1, max_retries + 1):

        try:

            with global_request_semaphore:  # Limit concurrent requests

                result = subprocess.run(command, shell=True, check=True, text=True)



            log_message(f"RunInfo for GDS ID {gds_id} saved to {output_file}")

            time.sleep(0.34)  # Ensure at most 3 requests per second

            return output_file

        except subprocess.CalledProcessError as e:

            log_message(f"Attempt {attempt}: Error fetching RunInfo for GDS ID {gds_id}: {e}", level="WARNING")

            if attempt < max_retries:

                backoff_time = delay * (2 ** (attempt - 1))

                log_message(f"Retrying in {backoff_time} seconds...")

                time.sleep(backoff_time)

            else:

                log_message(f"Max retries reached for GDS ID {gds_id}.", level="ERROR")

                return None




In [ ]:
#This Script is called GeoDataset.py

import os

import pandas as pd

from utils import fetch_numeric_id, fetch_runinfo, log_message, create_directory





class GeoDataset:

    def __init__(self, geo_accessions, email="your_email@example.com", tool="FetchOmics"):

        if isinstance(geo_accessions, str):

            self.geo_accessions = [geo_accessions]

        elif isinstance(geo_accessions, list):

            self.geo_accessions = geo_accessions

        else:

            raise ValueError("geo_accessions must be a string or a list of strings.")



        self.email = email

        self.tool = tool

        self.results = []  # Stores the processing results for all accessions

        self.metadata_summary = []  # Accumulates metadata for all runs

        self.srr_list = []  # Stores all SRR IDs (Run column)



    def parse_runinfo(self, runinfo_file, geo_dir):

        """

        Parse the SRA_RunInfo.csv file to extract metadata and save SRR.csv in the GEO subdirectory.

        Args:

            runinfo_file (str): Path to the SRA_RunInfo.csv file.

            geo_dir (str): Path to the GEO subdirectory.

        Returns:

            dict: Extracted metadata including species, LibraryStrategy, LibraryLayout, Platform, Model, and Runs.

        """

        try:

            df = pd.read_csv(runinfo_file)

            if df.empty:

                log_message(f"RunInfo file {runinfo_file} is empty.", level="WARNING")

                return None



            # Extract unique values from relevant columns

            species = df["ScientificName"].unique().tolist() if "ScientificName" in df.columns else []

            library_strategy = df["LibraryStrategy"].unique().tolist() if "LibraryStrategy" in df.columns else []

            library_layout = df["LibraryLayout"].unique().tolist() if "LibraryLayout" in df.columns else []

            platform = df["Platform"].unique().tolist() if "Platform" in df.columns else []

            model = df["Model"].unique().tolist() if "Model" in df.columns else []

            runs = df["Run"].tolist() if "Run" in df.columns else []



            # Save SRR.csv in the GEO subdirectory

            srr_file = os.path.join(geo_dir, "SRR.csv")

            srr_df = pd.DataFrame({"Run": runs})

            srr_df.to_csv(srr_file, index=False, header=True)

            log_message(f"SRR list saved to {srr_file}")



            # Append runs to the global SRR list

            self.srr_list.extend(runs)



            return {

                "species": species,

                "library_strategy": library_strategy,

                "library_layout": library_layout,

                "platform": platform,

                "model": model,

                "runs": runs,

            }

        except Exception as e:

            log_message(f"Error parsing RunInfo file {runinfo_file}: {e}", level="ERROR")

            return None



    def process_single_accession(self, accession, base_dir="output"):

        """

        Process a single GEO accession.

        Args:

            accession (str): GEO accession to process.

            base_dir (str): Base directory to store output files.

        Returns:

            dict: Process results for the GEO accession.

        """

        log_message(f"Processing GEO accession: {accession}")



        # Create directory for this accession

        geo_dir = create_directory(os.path.join(base_dir, accession))



        # Fetch GDS ID

        gds_id = fetch_numeric_id(accession, email=self.email, tool=self.tool)

        if not gds_id:

            log_message(f"No GDS ID found for GEO accession: {accession}", level="WARNING")

            return {

                "accession": accession,

                "gds_id": None,

                "geo_directory": geo_dir,

                "metadata": None,

            }



        # Fetch RunInfo file

        runinfo_file = fetch_runinfo(gds_id, output_dir=geo_dir)

        if not runinfo_file:

            log_message(f"RunInfo file not found for GEO accession: {accession}", level="ERROR")

            return {

                "accession": accession,

                "gds_id": gds_id,

                "geo_directory": geo_dir,

                "metadata": None,

            }



        # Parse RunInfo file for metadata and save SRR.csv

        metadata = self.parse_runinfo(runinfo_file, geo_dir)

        if metadata:

            # Add metadata to the summary

            for run in metadata["runs"]:

                self.metadata_summary.append({

                    "accession": accession,

                    "gds_id": gds_id,

                    "species": ", ".join(metadata["species"]),

                    "library_strategy": ", ".join(metadata["library_strategy"]),

                    "library_layout": ", ".join(metadata["library_layout"]),

                    "platform": ", ".join(metadata["platform"]),

                    "model": ", ".join(metadata["model"]),

                    "run": run,

                })



        # Return results

        result = {

            "accession": accession,

            "gds_id": gds_id,

            "geo_directory": geo_dir,

            "metadata": metadata,

        }

        log_message(f"Finished processing GEO accession: {accession}")

        return result



    def process_all(self, base_dir="output", summary_csv="metadata_summary.csv"):

        """

        Process all GEO accessions and generate a metadata summary.

        Args:

            base_dir (str): Base directory to store output files.

            summary_csv (str): Path to save the metadata summary CSV.

        Returns:

            list: List of results for each GEO accession.

        """

        log_message(f"Starting batch processing of {len(self.geo_accessions)} GEO accessions.")

        for accession in self.geo_accessions:

            result = self.process_single_accession(accession, base_dir)

            self.results.append(result)



        # Save metadata summary CSV

        if self.metadata_summary:

            summary_df = pd.DataFrame(self.metadata_summary)

            summary_df.to_csv(summary_csv, index=False)

            log_message(f"Metadata summary saved to {summary_csv}")



        log_message("Finished batch processing.")

        return self.results




In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[ ]:
#This script is called SRRProcessor.py

import os
import pandas as pd
import subprocess
from utils import log_message, create_directory


class SRRProcessor:
    def __init__(self):
        """
        Initialize the SRRProcessor.

        Assumes that `prefetch` and `fasterq-dump` are available in the global PATH.
        """
        pass  # No need for paths if tools are globally accessible

    def process_srrs_in_geo_dirs(self, geo_results):
        """
        Process SRRs dynamically from the `geo_dir` paths created by GeoDataset.

        Args:
            geo_results (list): The results list from GeoDataset's `process_all` method.

        Returns:
            None
        """
        for result in geo_results:
            accession = result["accession"]
            geo_dir = result["geo_directory"]
            srr_csv_path = os.path.join(geo_dir, "SRR.csv")
            runinfo_path = os.path.join(geo_dir, "SRA_RunInfo.csv")

            # Check if SRR.csv exists in the GEO directory
            if not os.path.exists(srr_csv_path):
                log_message(f"SRR.csv not found in {geo_dir}. Skipping {accession}.", level="WARNING")
                continue

            # Check if RunInfo.csv exists
            if not os.path.exists(runinfo_path):
                log_message(f"RunInfo.csv not found in {geo_dir}. Skipping {accession}.", level="WARNING")
                continue

            # Read SRRs from SRR.csv
            try:
                srr_df = pd.read_csv(srr_csv_path)
                srr_ids = srr_df["Run"].tolist()
                if not srr_ids:
                    log_message(f"No SRRs found in {srr_csv_path}. Skipping {accession}.", level="WARNING")
                    continue
            except Exception as e:
                log_message(f"Error reading {srr_csv_path}: {e}", level="ERROR")
                continue

            # Read LibraryLayout from RunInfo.csv
            try:
                runinfo_df = pd.read_csv(runinfo_path)
                layout_map = {
                    row["Run"]: row["LibraryLayout"] if "LibraryLayout" in row else "SINGLE"
                    for _, row in runinfo_df.iterrows()
                }
            except Exception as e:
                log_message(f"Error reading {runinfo_path}: {e}", level="ERROR")
                continue

            # Process SRRs
            self._process_srr_list(srr_ids, geo_dir, layout_map)

    def _process_srr_list(self, srr_ids, geo_dir, layout_map):
        """
        Helper method to process a list of SRR IDs.

        Args:
            srr_ids (list): List of SRR IDs to process.
            geo_dir (str): Base GEO directory where SRR subdirectories will be created.
            layout_map (dict): Mapping of SRR IDs to LibraryLayout (PAIRED or SINGLE).

        Returns:
            None
        """
        for srr_id in srr_ids:
            log_message(f"Processing SRR: {srr_id}")

            # Create a subdirectory for each SRR within the GEO directory
            srr_output_dir = create_directory(os.path.join(geo_dir, f"{srr_id}_Fastq"))

            try:
                # Step 1: Prefetch the SRR file
                log_message(f"Fetching SRR {srr_id}...")
                prefetch_command = ["prefetch", srr_id, "-O", srr_output_dir]
                subprocess.run(prefetch_command, check=True, text=True)
                log_message(f"Successfully fetched {srr_id}.")

                # Step 2: Convert to FASTQ format
                log_message(f"Converting {srr_id} to FASTQ...")
                sra_file = os.path.join(srr_output_dir, f"{srr_id}.sra")
                fasterq_command = ["fasterq-dump", "--split-files", sra_file, "-O", srr_output_dir]
                subprocess.run(fasterq_command, check=True, text=True)

                # Handle paired-end vs single-end naming
                layout = layout_map.get(srr_id, "SINGLE").upper()
                if layout == "PAIRED":
                    log_message(f"Detected PAIRED layout for {srr_id}.")
                    # Files remain `_1.fastq` and `_2.fastq`
                else:
                    log_message(f"Detected SINGLE layout for {srr_id}.")
                    single_fastq = os.path.join(srr_output_dir, f"{srr_id}_1.fastq")
                    if os.path.exists(single_fastq):
                        os.rename(single_fastq, os.path.join(srr_output_dir, f"{srr_id}.fastq"))

                # Step 3: Clean up the SRA file (optional)
                if os.path.exists(sra_file):
                    os.remove(sra_file)
                    log_message(f"Deleted SRA file for {srr_id}.")

            except subprocess.CalledProcessError as e:
                log_message(f"Error processing {srr_id}: {e}", level="ERROR")
                continue

        log_message(f"Finished processing SRRs in {geo_dir}.")



In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[ ]:
#This is called main.py. It is the main script associated with the package that is being developed.

import sys
import os
from GeoDataset import GeoDataset
from SRRProcessor import SRRProcessor
from utils import log_message

def main():
    if len(sys.argv) < 3:
        print("Usage: python main.py <geo_accession_or_file> <base_output_dir> [email] [tool]")
        print("Example: python main.py GSE113046 /path/to/output/directory your_email@example.com FetchOmics")
        print("Example: python main.py geo_accessions.txt /path/to/output/directory your_email@example.com FetchOmics")
        print("\nThis script processes GEO accessions to fetch metadata and convert SRR files into FASTQ files.")
        sys.exit(1)

    geo_input = sys.argv[1]
    base_output_dir = sys.argv[2]

    # Optional arguments
    email = sys.argv[3] if len(sys.argv) > 3 else "your_email@example.com"
    tool = sys.argv[4] if len(sys.argv) > 4 else "FetchOmics"

    # Determine if input is a file or a single accession
    if os.path.exists(geo_input):
        # Input is a file
        try:
            with open(geo_input, "r") as f:
                geo_accessions = [line.strip() for line in f if line.strip()]
        except Exception as e:
            print(f"Error reading '{geo_input}': {e}")
            sys.exit(1)

        if not geo_accessions:
            print("Error: No GEO accessions found in the input file. Please provide at least one GEO accession.")
            sys.exit(1)

    else:
        # Input is a single GEO accession
        geo_accessions = [geo_input]

    print(f"Processing the following GEO accessions: {geo_accessions}")
    print(f"Using email: {email}, tool: {tool}")

    geo = GeoDataset(geo_accessions, email=email, tool=tool)

    try:
        geo_results = geo.process_all(base_dir=base_output_dir, summary_csv=os.path.join(base_output_dir, "metadata_summary.csv"))
    except Exception as e:
        log_message(f"An error occurred during GEO accession processing: {e}", level="ERROR")
        sys.exit(1)

    srr_processor = SRRProcessor()

    try:
        log_message("Starting SRR processing...")
        srr_processor.process_srrs_in_geo_dirs(geo_results)
        log_message("SRR processing completed successfully.")
    except Exception as e:
        log_message(f"An error occurred during SRR processing: {e}", level="ERROR")
        sys.exit(1)

    print("FetchOmics processing completed successfully.")

if __name__ == "__main__":
    main()



In [ ]:
#!/bin/bash
# In[ ]:
#This is a .slurm file for utilizing the HPC.

#SBATCH --job-name=FetchOmics             # Job name

#SBATCH --account=sihogan0                # Account name

#SBATCH --partition=standard              # Partition/queue name

#SBATCH --nodes=1                         # Number of nodes

#SBATCH --ntasks=1                        # Single task per job

#SBATCH --cpus-per-task=8                 # Number of CPU cores per task

#SBATCH --mem=32G                         # Memory allocation

#SBATCH --time=04:00:00                   # Maximum runtime

#SBATCH --output=/nfs/turbo/umms-sihogan/crizza/logs/fetchomics_%A.log # Centralized log file

#SBATCH --error=/nfs/turbo/umms-sihogan/crizza/logs/fetchomics_%A.log  # Redirect errors to the same file

#SBATCH --array=0-63%5                    # Array of 64 tasks, 5 running concurrently



# Load necessary modules

module purge

module load Bioinformatics

module load sratoolkit

module load python/3.12.1


# Input file and output directory

GEO_LIST="/nfs/turbo/umms-sihogan/crizza/FetchOmics/geo_accessions.txt"

OUTPUT_DIR="/nfs/turbo/umms-sihogan/crizza"



# Get the current GEO accession based on the array task ID

ACCESSION=$(sed -n "$((SLURM_ARRAY_TASK_ID + 1))p" "$GEO_LIST")



# Check if ACCESSION is empty

if [ -z "$ACCESSION" ]; then

    echo "No GEO accession found for task ID $SLURM_ARRAY_TASK_ID"

    exit 1

fi



# Run the main script for this accession

cd /nfs/turbo/umms-sihogan/crizza/FetchOmics && python main.py "$ACCESSION" "$OUTPUT_DIR"


